# Healthcare Analytics Demo — One-Click Launcher

**Run All** to deploy the complete Healthcare Payer/Provider analytics solution into this Fabric workspace.

### What Gets Deployed
| Layer | Items |
|-------|-------|
| **Lakehouses** | `lh_bronze_raw`, `lh_silver_stage`, `lh_silver_ods`, `lh_gold_curated` |
| **Notebooks** | 5 ETL + 2 data generation + 5 RTI (event simulator, Eventhouse setup, 3 scoring) |
| **Pipelines** | `PL_Healthcare_Full_Load`, `PL_Healthcare_Master` |
| **Semantic Model** | `HealthcareDemoHLS` (star schema) |
| **Data Agent** | `HealthcareHLSAgent` (Copilot AI) |
| **Ontology + Graph** | `Healthcare_Demo_Ontology_HLS` + `Healthcare_Demo_Graph` (deployed via API after pipeline) |
| **Real-Time Intelligence** | Eventhouse + KQL DB + 3 scoring use cases (fraud, care gaps, high-cost trajectory) |

### Prerequisites
- An **empty** Fabric workspace (this notebook should be the only item)
- Fabric capacity: **F64** or higher recommended
- User must be workspace **Admin** or **Member**

### Configuration
Edit the cell below to point to your GitHub repo (public or private).

In [ ]:
# ============================================================================
# CELL 1 — Install fabric-launcher
# ============================================================================

%pip install fabric-launcher --quiet

import notebookutils
from fabric_launcher import FabricLauncher

print("fabric-launcher installed successfully")

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values
# ============================================================================

GITHUB_OWNER = "rasgiza"                # GitHub org or user (public mirror)
GITHUB_REPO  = "Fabric-Payer-Provider-HealthCare-Demo"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                    # Leave empty for public repos. Only needed for private repos (classic PAT with repo scope).

# Deployment options
GENERATE_DATA = True                  # Generate fresh synthetic data
RUN_PIPELINE = True                   # Run the full-load pipeline after data gen
UPLOAD_KNOWLEDGE_DOCS = True          # Upload healthcare knowledge docs to lakehouse
DEPLOY_RTI = True                     # Deploy Real-Time Intelligence (Eventhouse + scoring)

print(f"Will deploy from: github.com/{GITHUB_OWNER}/{GITHUB_REPO} @ {GITHUB_BRANCH}")

In [ ]:
# ============================================================================
# CELL 2 — Initialize launcher
# ============================================================================

launcher = FabricLauncher(
    notebookutils,
    environment="DEV",
    allow_non_empty_workspace=False,
    fix_zero_logical_ids=True,
    debug=False,
)

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")
print(f"Workspace ID: {workspace_id}")
print(f"Launcher ready")

In [ ]:
# ============================================================================
# CELL 3 — Download repo & deploy all Fabric artifacts from scratch (staged)
# ============================================================================
# Downloads the repo as a ZIP, extracts it, and creates every artifact
# in the workspace using the Fabric REST API.  Order matters because
# later items reference earlier ones via logicalId.
#
# Stage 1: Lakehouses          — must exist before notebooks reference them
# Stage 2: Eventhouse           — async provisioning, needs its own stage
# Stage 3: KQL Database         — depends on Eventhouse being fully ready
# Stage 4: Notebooks (create)  — creates notebook items in workspace
# Stage 5: Notebooks (content) — updateDefinition applies code reliably
#           (Fabric Create Item API can silently drop inline definitions
#            for notebooks; a second pass forces updateDefinition which
#            always applies the content.)
# Stage 6: Data Pipelines      — orchestrate notebook execution
# Stage 7: Data Agent     — SM created later (Cell 9) after Gold tables exist
# ============================================================================

print("Downloading repo and deploying artifacts...")
print("This takes 3-5 minutes.\n")

downloader, deployer, report = launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    github_token=GITHUB_TOKEN if GITHUB_TOKEN else None,
    workspace_folder="workspace",
    item_type_stages=[
        ["Lakehouse"],                         # Stage 1
        ["Eventhouse"],                        # Stage 2 — provision Eventhouse first
        ["KQLDatabase"],                       # Stage 3 — KQL DB needs Eventhouse ready
        ["Notebook"],                          # Stage 4 — create notebook items
        ["Notebook"],                          # Stage 5 — updateDefinition ensures code is applied
        ["DataPipeline"],                      # Stage 6
        ["DataAgent"],                          # Stage 7 (SM created in Cell 9 after pipeline)
    ],
    validate_after_deployment=True,
    generate_report=True,
    deployment_retries=2,
)

print("\n" + "=" * 60)
print("  ARTIFACT DEPLOYMENT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 3b -- Push notebook content via ipynb format (guaranteed to work)
# ============================================================================
# The Fabric Create Item API silently creates empty notebook shells.
# The generic updateDefinition with .py format returns 200 but doesn't apply.
# Solution: Convert .py -> .ipynb format and push via the notebook-specific
# updateDefinition API, which documents ipynb as the supported format.
# ============================================================================

import base64, requests, os, json, time, re

print("Converting notebooks to ipynb and pushing via REST API...\n")

token = notebookutils.credentials.getToken("pbi")
hdrs = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# -- Helper: Convert Fabric .py source -> ipynb JSON ----------------------
def py_to_ipynb(py_content, metadata_json=None, logical_to_real=None, ws_id=None):
    """Convert Fabric notebook .py source to Jupyter ipynb JSON."""
    # Replace %pip magic with subprocess (disabled in child notebooks via notebookutils.notebook.run)
    py_content = re.sub(
        r'^%pip install (.+)$',
        r'import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install"] + "\1".split())',
        py_content,
        flags=re.MULTILINE,
    )
    lines = py_content.split("\n")
    cells = []
    i = 0

    # Skip header
    while i < len(lines) and (lines[i].strip() in ("", "# Fabric notebook source")):
        i += 1

    while i < len(lines):
        # Look for METADATA marker
        if lines[i].startswith("# METADATA **"):
            i += 1
            while i < len(lines) and lines[i].strip() == "":
                i += 1
            if i >= len(lines):
                break

            # Determine cell type
            if lines[i].startswith("# CELL **"):
                cell_type = "code"
            elif lines[i].startswith("# MARKDOWN **"):
                cell_type = "markdown"
            else:
                i += 1
                continue

            i += 1
            # Skip blank line after marker
            if i < len(lines) and lines[i].strip() == "":
                i += 1

            # Collect content until next METADATA or EOF
            cell_lines = []
            while i < len(lines) and not lines[i].startswith("# METADATA **"):
                cell_lines.append(lines[i])
                i += 1

            # Remove trailing blank lines
            while cell_lines and cell_lines[-1].strip() == "":
                cell_lines.pop()

            # For markdown: remove "# " prefix
            if cell_type == "markdown":
                processed = []
                for ln in cell_lines:
                    if ln.startswith("# "):
                        processed.append(ln[2:])
                    elif ln == "#":
                        processed.append("")
                    else:
                        processed.append(ln)
                cell_lines = processed

            if cell_lines:
                # Build source array (each line gets \n except the last)
                source = [ln + "\n" for ln in cell_lines[:-1]]
                source.append(cell_lines[-1])

                nb_cell = {"cell_type": cell_type, "metadata": {}, "source": source}
                if cell_type == "code":
                    nb_cell["outputs"] = []
                    nb_cell["execution_count"] = None
                cells.append(nb_cell)
        else:
            i += 1

    # Build notebook structure
    nb_metadata = {
        "kernel_info": {"name": "synapse_pyspark"},
        "kernelspec": {
            "name": "synapse_pyspark",
            "display_name": "Synapse PySpark",
            "language": "Python",
        },
        "language_info": {"name": "python"},
    }

    # Merge lakehouse dependencies from metadata JSON
    if metadata_json:
        deps = metadata_json.get("dependencies", {})
        if deps:
            # Replace logical IDs in the metadata
            deps_str = json.dumps(deps)
            if logical_to_real:
                for lid, rid in logical_to_real.items():
                    deps_str = deps_str.replace(lid, rid)
            if ws_id:
                deps_str = deps_str.replace("00000000-0000-0000-0000-000000000000", ws_id)
            nb_metadata["dependencies"] = json.loads(deps_str)

    return json.dumps(
        {"nbformat": 4, "nbformat_minor": 5, "metadata": nb_metadata, "cells": cells},
        indent=1,
    )

# -- Step 1: Get all deployed items --------------------------------------
resp = requests.get(f"{api}/items", headers=hdrs)
all_items = resp.json().get("value", [])
name_to_id = {(it["type"], it["displayName"]): it["id"] for it in all_items}

# -- Step 2: Locate extracted workspace directory ------------------------
ws_dir = None
for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    # Direct path
    d = os.path.join(base, "workspace")
    if os.path.isdir(d):
        ws_dir = d
        break
    # One level deep (fabric-launcher extracts ZIP to a subdirectory)
    if os.path.isdir(base):
        for sub in os.listdir(base):
            d = os.path.join(base, sub, "workspace")
            if os.path.isdir(d):
                ws_dir = d
                break
    if ws_dir:
        break

if not ws_dir:
    print("ERROR: Could not find extracted workspace directory.")
    print("  Searched .lakehouse/default/Files/src/ (direct and one level deep)")
    if os.path.isdir(".lakehouse/default/Files/src"):
        print("  Contents of Files/src/:", os.listdir(".lakehouse/default/Files/src"))
    elif os.path.isdir("/lakehouse/default/Files/src"):
        print("  Contents of Files/src/:", os.listdir("/lakehouse/default/Files/src"))
else:
    # Build logical_id -> real_id mapping
    logical_to_real = {}
    for entry in os.listdir(ws_dir):
        pf = os.path.join(ws_dir, entry, ".platform")
        if os.path.exists(pf):
            with open(pf, "r") as f:
                plat = json.load(f)
            itype = plat["metadata"]["type"]
            iname = plat["metadata"]["displayName"]
            lid = plat["config"]["logicalId"]
            rid = name_to_id.get((itype, iname), "")
            if lid and rid:
                logical_to_real[lid] = rid

    # -- Step 3: Convert and push each notebook --------------------------
    pushed, failed = 0, 0
    for entry in sorted(os.listdir(ws_dir)):
        if not entry.endswith(".Notebook"):
            continue
        nb_name = entry.replace(".Notebook", "")
        nb_id = name_to_id.get(("Notebook", nb_name))
        if not nb_id:
            print(f"  SKIP {nb_name}: not in workspace")
            continue

        nb_path = os.path.join(ws_dir, entry)

        # Read .py content
        py_file = os.path.join(nb_path, "notebook-content.py")
        if not os.path.exists(py_file):
            print(f"  SKIP {nb_name}: no notebook-content.py")
            continue
        with open(py_file, "r", encoding="utf-8") as f:
            py_content = f.read()

        # Read metadata JSON
        meta_file = os.path.join(nb_path, ".metadata", "notebook-metadata.json")
        meta_json = None
        if os.path.exists(meta_file):
            with open(meta_file, "r", encoding="utf-8") as f:
                meta_json = json.load(f)

        # Convert to ipynb
        ipynb_content = py_to_ipynb(py_content, meta_json, logical_to_real, workspace_id)
        ipynb_b64 = base64.b64encode(ipynb_content.encode("utf-8")).decode("utf-8")

        # Build payload -- single part: the ipynb file
        parts = [
            {"path": "notebook-content.ipynb", "payload": ipynb_b64, "payloadType": "InlineBase64"}
        ]
        body = {"definition": {"format": "ipynb", "parts": parts}}

        # POST to updateDefinition (with retry for transient 500s)
        r = None
        for attempt in range(3):
            r = requests.post(
                f"{api}/items/{nb_id}/updateDefinition",
                headers=hdrs,
                json=body,
            )
            if r.status_code in (200, 202):
                break
            if r.status_code >= 500 and attempt < 2:
                print(f"  {nb_name}: HTTP {r.status_code}, retrying ({attempt+1}/3)...")
                time.sleep(5 * (attempt + 1))
            else:
                break

        # Handle LRO (202)
        if r.status_code == 202:
            loc = r.headers.get("Location", "")
            retry_after = int(r.headers.get("Retry-After", "3"))
            if loc:
                for _ in range(60):
                    time.sleep(retry_after)
                    pr = requests.get(loc, headers=hdrs)
                    if pr.status_code == 200:
                        break
                    elif pr.status_code != 202:
                        break

        if r.status_code in (200, 202):
            # Count cells for feedback
            cell_count = ipynb_content.count('"cell_type"')
            size_kb = round(len(ipynb_content) / 1024, 1)
            pushed += 1
            print(f"  {nb_name}: OK ({cell_count} cells, {size_kb}KB)")
        else:
            failed += 1
            print(f"  FAIL {nb_name}: HTTP {r.status_code} -- {r.text[:300]}")

    print(f"\n{'='*60}")
    print(f"  Notebooks pushed: {pushed} (ipynb format)")
    if failed:
        print(f"  Failed: {failed}")

    # -- Step 4: Verify --------------------------------------------------
    print(f"\nVerifying content (getDefinition on 01_Bronze_Ingest_CSV)...")
    test_id = name_to_id.get(("Notebook", "01_Bronze_Ingest_CSV"))
    if test_id:
        vr = requests.post(f"{api}/items/{test_id}/getDefinition", headers=hdrs, json={})
        # Handle LRO for getDefinition
        if vr.status_code == 202:
            loc = vr.headers.get("Location", "")
            if loc:
                for _ in range(30):
                    time.sleep(2)
                    vr = requests.get(loc, headers=hdrs)
                    if vr.status_code == 200:
                        break

        if vr.status_code == 200:
            vdata = vr.json()
            defn_parts = vdata.get("definition", {}).get("parts", [])
            print(f"  Parts returned: {[p.get('path') for p in defn_parts]}")
            for p in defn_parts:
                if "content" in p.get("path", "").lower():
                    decoded = base64.b64decode(p["payload"]).decode("utf-8")
                    has_spark = "spark" in decoded.lower() or "bronze" in decoded.lower()
                    size = len(decoded)
                    print(f"  Content size: {size:,} bytes, has_code: {has_spark}")
                    if size > 500:
                        print(f"  [OK] Notebook content verified -- has real code")
                    else:
                        print(f"  [!] Content is suspiciously small: {decoded[:200]}")
                    break
            else:
                print(f"  [!] No content part found. All parts: {json.dumps([{k:v for k,v in p.items() if k != 'payload'} for p in defn_parts], indent=2)}")
        else:
            print(f"  getDefinition returned HTTP {vr.status_code}: {vr.text[:200]}")
    print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 4 — Upload healthcare knowledge docs to Data Agent lakehouse
# ============================================================================

if UPLOAD_KNOWLEDGE_DOCS:
    import os, glob

    # fabric-launcher extracts the repo ZIP to this relative path by default
    # Try both relative (library default) and absolute (Fabric mount) paths
    for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(candidate):
            extract_base = candidate
            break
    else:
        print("Warning: Could not find extract directory. Skipping knowledge doc upload.")
        print("  Tried: .lakehouse/default/Files/src and /lakehouse/default/Files/src")
        UPLOAD_KNOWLEDGE_DOCS = False
        extract_base = None

    # fabric-launcher extracts repo contents directly into extract_base (no wrapper folder)
    repo_root = extract_base

    knowledge_src = os.path.join(repo_root, "healthcare_knowledge") if repo_root else None

    if knowledge_src and os.path.exists(knowledge_src):
        print("Uploading healthcare knowledge documents...")
        launcher.upload_files_to_lakehouse(
            lakehouse_name="lh_gold_curated",
            source_directory=knowledge_src,
            target_folder="healthcare_knowledge",
            file_patterns=["*.md"],
        )
        # Count uploaded files
        count = sum(len(files) for _, _, files in os.walk(knowledge_src) if files)
        print(f"Uploaded {count} knowledge documents to lh_gold_curated/Files/healthcare_knowledge/")
    else:
        print(f"Warning: healthcare_knowledge folder not found at {knowledge_src}")
else:
    print("Skipping knowledge doc upload (UPLOAD_KNOWLEDGE_DOCS=False)")

In [ ]:
# ============================================================================
# CELL 5 — Generate synthetic data
# ============================================================================
# Uses notebookutils.notebook.run() — the native Fabric way to orchestrate
# notebooks. More reliable than the Jobs REST API after updateDefinition.
# ============================================================================

if GENERATE_DATA:
    print("Running NB_Generate_Sample_Data (generates ~10K patients, 100K encounters)...")
    print("This takes 2-4 minutes.\n")

    try:
        notebookutils.notebook.run("NB_Generate_Sample_Data", 1200, {"useRootDefaultLakehouse": True})
        print("\n✅ Data generation SUCCEEDED")
    except Exception as e:
        print(f"\n❌ Data generation FAILED: {e}")
        print("Try running NB_Generate_Sample_Data manually from the workspace.")
else:
    print("Skipping data generation (GENERATE_DATA=False)")

In [ ]:
# ============================================================================
# CELL 6 — Run the full-load pipeline
# ============================================================================

if RUN_PIPELINE:
    import requests, time, json

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the pipeline ID
    print("Looking up PL_Healthcare_Master pipeline...")
    resp = requests.get(f"{api_base}/items?type=DataPipeline", headers=headers)
    resp.raise_for_status()
    pipelines = resp.json().get("value", [])
    pipeline = next((p for p in pipelines if p["displayName"] == "PL_Healthcare_Master"), None)

    if not pipeline:
        print("WARNING: PL_Healthcare_Master not found. Skipping pipeline run.")
        print("You can run it manually from the workspace.")
    else:
        pipeline_id = pipeline["id"]
        print(f"Pipeline ID: {pipeline_id}")

        # Trigger pipeline with load_mode=full
        print("Triggering pipeline with load_mode=full...")
        trigger_body = {
            "executionData": {
                "parameters": {
                    "load_mode": "full"
                }
            }
        }
        resp = requests.post(
            f"{api_base}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
            headers=headers,
            json=trigger_body,
        )

        if resp.status_code in (200, 202):
            # Get job ID from Location header or response
            location = resp.headers.get("Location", "")
            print(f"Pipeline triggered. Polling for completion...")
            print(f"(This takes 8-15 minutes for full load)\n")

            # Poll until complete
            max_polls = 120  # 120 * 15s = 30 min max
            for poll in range(max_polls):
                time.sleep(15)
                try:
                    if location:
                        poll_resp = requests.get(location, headers=headers)
                    else:
                        poll_resp = requests.get(
                            f"{api_base}/items/{pipeline_id}/jobs/instances",
                            headers=headers,
                        )
                    if poll_resp.status_code == 200:
                        job_data = poll_resp.json()
                        status = job_data.get("status", "Unknown")
                        if status in ("Completed", "Succeeded"):
                            print(f"  Pipeline COMPLETED after {(poll+1)*15}s")
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  Pipeline {status} after {(poll+1)*15}s")
                            print(f"  Check the pipeline run history for details.")
                            break
                        else:
                            if poll % 4 == 0:  # Print every 60s
                                print(f"  [{(poll+1)*15}s] Status: {status}...")
                except Exception as e:
                    if poll % 4 == 0:
                        print(f"  [{(poll+1)*15}s] Polling... ({e})")
            else:
                print("  Pipeline still running after 30 min. Check workspace for status.")
        else:
            print(f"  Pipeline trigger returned {resp.status_code}: {resp.text}")
            print("  You can run it manually from the workspace.")
else:
    print("Skipping pipeline run (RUN_PIPELINE=False)")
    print("To run manually: Open PL_Healthcare_Master → Run with load_mode=full")

In [ ]:
# ============================================================================
# CELL 6b -- Create & Refresh Semantic Model (Direct Lake)
# ============================================================================
# The SM is NOT deployed via fabric-launcher (placeholder GUIDs in the repo
# would corrupt AS internal metadata). Instead we create it here, AFTER the
# pipeline has run and Gold tables exist in lh_gold_curated.
#
# Steps:
#   1. Find lh_gold_curated lakehouse ID
#   2. Delete any existing SM with the same name (idempotent re-run)
#   3. Load all TMDL files from the extracted repo on the lakehouse filesystem
#   4. Patch expressions.tmdl URL with correct workspace/lakehouse IDs
#   5. Create fresh SM via POST /semanticModels with all definition parts
#   6. Trigger Full refresh
# ============================================================================

import requests, time, base64, re, json, os

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
FABRIC_API = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

SM_NAME = "HealthcareDemoHLS"
SM_REPO_DIR = "workspace/HealthcareDemoHLS.SemanticModel"
URL_PATTERN = re.compile(
    r'https://onelake\.dfs\.fabric\.microsoft\.com/'
    r'[0-9a-fA-F-]{36}/[0-9a-fA-F-]{36}'
)

def wait_lro(resp, hdr, timeout=120):
    """Poll an LRO and return (final_status, result_body)."""
    loc = resp.headers.get("Location", "")
    retry = int(resp.headers.get("Retry-After", 5))
    elapsed = 0
    while elapsed < timeout:
        time.sleep(retry)
        elapsed += retry
        r = requests.get(loc, headers=hdr)
        if r.status_code != 200:
            continue
        body = r.json()
        st = body.get("status", "")
        if st == "Succeeded":
            res_loc = body.get("resourceLocation", "")
            if res_loc:
                rr = requests.get(res_loc, headers=hdr)
                if rr.status_code == 200:
                    return "Succeeded", rr.json()
            rr = requests.get(f"{loc}/result", headers=hdr)
            if rr.status_code == 200:
                return "Succeeded", rr.json()
            return "Succeeded", body
        elif st in ("Failed", "Cancelled"):
            err = body.get("error", {})
            return st, err
    return "Timeout", {}

print("=" * 60)
print("  SEMANTIC MODEL -- Create from Repo Definition")
print("=" * 60)

# -- Step 1: Discover lakehouse ID ----------------------------------------
resp = requests.get(f"{FABRIC_API}/lakehouses", headers=headers)
resp.raise_for_status()
lh_gold_id = None
for lh in resp.json().get("value", []):
    if lh["displayName"] == "lh_gold_curated":
        lh_gold_id = lh["id"]
        break

if not lh_gold_id:
    print("  [WARN] lh_gold_curated not found -- skipping SM creation")
    print("  Run the pipeline first, then re-run this cell.")
else:
    print(f"  Lakehouse: lh_gold_curated ({lh_gold_id})")
    print(f"  Workspace: {workspace_id}")
    new_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lh_gold_id}"

    # -- Step 2: Delete existing SM if any (idempotent re-run) -------------
    resp = requests.get(f"{FABRIC_API}/items?type=SemanticModel", headers=headers)
    resp.raise_for_status()
    for sm in resp.json().get("value", []):
        if sm["displayName"] == SM_NAME:
            old_id = sm["id"]
            print(f"  Deleting existing SM: {SM_NAME} ({old_id})...")
            dr = requests.delete(f"{FABRIC_API}/semanticModels/{old_id}", headers=headers)
            if dr.status_code in (200, 202, 204):
                if dr.status_code == 202:
                    wait_lro(dr, headers, timeout=60)
                print(f"  Deleted.")
            else:
                print(f"  [WARN] Delete HTTP {dr.status_code}: {dr.text[:200]}")
            print("  Waiting 10s for name availability...")
            time.sleep(10)
            break

    # -- Step 3: Load TMDL from extracted repo on lakehouse ----------------
    # fabric-launcher extracts to .lakehouse/default/Files/src/<repodir>/
    sm_base = None
    for base_prefix in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if not os.path.isdir(base_prefix):
            continue
        # Direct path
        direct = os.path.join(base_prefix, SM_REPO_DIR)
        if os.path.isdir(direct):
            sm_base = direct
            break
        # One level down (extracted repo subdirectory)
        for sub in os.listdir(base_prefix):
            nested = os.path.join(base_prefix, sub, SM_REPO_DIR)
            if os.path.isdir(nested):
                sm_base = nested
                break
        if sm_base:
            break

    if not sm_base:
        print("  [FAIL] SM definition directory not found on lakehouse filesystem.")
        print("  Expected: Files/src/[<repo-dir>/]workspace/HealthcareDemoHLS.SemanticModel/")
        # List what's actually there for debugging
        for base_prefix in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
            if os.path.isdir(base_prefix):
                print(f"  Contents of {base_prefix}:")
                for item in os.listdir(base_prefix)[:10]:
                    print(f"    {item}")
    else:
        print(f"  SM definition dir: {sm_base}")
        def_dir = os.path.join(sm_base, "definition")

        parts = []

        # Load definition.pbism
        pbism_path = os.path.join(sm_base, "definition.pbism")
        if os.path.exists(pbism_path):
            with open(pbism_path, "rb") as f:
                raw = f.read()
            if raw.startswith(b"\xef\xbb\xbf"):
                raw = raw[3:]
            parts.append({
                "path": "definition.pbism",
                "payload": base64.b64encode(raw).decode(),
                "payloadType": "InlineBase64"
            })

        # Load all definition/*.tmdl files (recursively for tables/)
        if os.path.isdir(def_dir):
            for root, dirs, files in os.walk(def_dir):
                for fname in sorted(files):
                    fpath = os.path.join(root, fname)
                    rel = "definition/" + os.path.relpath(fpath, def_dir).replace("\\", "/")
                    with open(fpath, "rb") as f:
                        raw = f.read()
                    if raw.startswith(b"\xef\xbb\xbf"):
                        raw = raw[3:]

                    # Patch expressions.tmdl URL
                    if "expressions.tmdl" in rel:
                        content = raw.decode("utf-8")
                        matches = URL_PATTERN.findall(content)
                        if matches and matches[0] != new_url:
                            print(f"  Patching URL: {matches[0]} -> {new_url}")
                            content = URL_PATTERN.sub(new_url, content)
                            raw = content.encode("utf-8")
                        elif matches:
                            print(f"  URL already correct in expressions.tmdl")
                        else:
                            print(f"  [WARN] No URL found in expressions.tmdl -- injecting correct URL")

                    parts.append({
                        "path": rel,
                        "payload": base64.b64encode(raw).decode(),
                        "payloadType": "InlineBase64"
                    })

        print(f"  Loaded {len(parts)} definition parts:")
        for p in parts[:5]:
            print(f"    {p['path']}")
        if len(parts) > 5:
            print(f"    ... and {len(parts) - 5} more")

        if not parts:
            print("  [FAIL] No definition parts loaded. Cannot create SM.")
        else:
            # -- Step 4: Create SM with correct definition -----------------
            token = notebookutils.credentials.getToken("pbi")
            headers["Authorization"] = f"Bearer {token}"

            create_body = {
                "displayName": SM_NAME,
                "description": "Healthcare Demo Direct Lake semantic model",
                "definition": {"parts": parts}
            }

            print(f"  Creating SM: {SM_NAME} with {len(parts)} TMDL parts...")
            r = requests.post(f"{FABRIC_API}/semanticModels", headers=headers, json=create_body)
            print(f"  Create HTTP {r.status_code}")

            sm_id = None
            if r.status_code in (200, 201):
                sm_id = r.json().get("id")
                print(f"  Created: {sm_id}")
            elif r.status_code == 202:
                st, body = wait_lro(r, headers, timeout=180)
                if st == "Succeeded":
                    time.sleep(3)
                    resp2 = requests.get(f"{FABRIC_API}/items?type=SemanticModel", headers=headers)
                    for sm in resp2.json().get("value", []):
                        if sm["displayName"] == SM_NAME:
                            sm_id = sm["id"]
                            break
                if sm_id:
                    print(f"  Created (async): {sm_id}")
                else:
                    print(f"  Create LRO {st}: {body}")
            else:
                print(f"  [FAIL] Create SM: HTTP {r.status_code}")
                print(f"  Response: {r.text[:500]}")
                sm_id = None

            # -- Step 5: Trigger refresh -----------------------------------
            if sm_id:
                print("  Waiting 15s for SM initialization...")
                time.sleep(15)

                token = notebookutils.credentials.getToken("pbi")
                headers["Authorization"] = f"Bearer {token}"

                pbi_base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}"
                refresh_url = f"{pbi_base}/datasets/{sm_id}/refreshes"

                MAX_ATTEMPTS = 3
                for attempt in range(1, MAX_ATTEMPTS + 1):
                    print(f"  Triggering refresh (attempt {attempt}/{MAX_ATTEMPTS})...")
                    r = requests.post(refresh_url, headers=headers, json={"type": "Full"})

                    if r.status_code not in (200, 202):
                        print(f"  Refresh trigger HTTP {r.status_code}: {r.text[:300]}")
                        if attempt < MAX_ATTEMPTS:
                            print(f"  Retrying in 15s...")
                            time.sleep(15)
                            token = notebookutils.credentials.getToken("pbi")
                            headers["Authorization"] = f"Bearer {token}"
                            continue
                        else:
                            print("  All attempts failed. Refresh manually from the workspace.")
                            break

                    # Poll for completion
                    success = False
                    for poll_i in range(60):
                        time.sleep(10)
                        poll_resp = requests.get(refresh_url, headers=headers)
                        if poll_resp.status_code == 200:
                            refreshes = poll_resp.json().get("value", [])
                            if refreshes:
                                latest = refreshes[0]
                                status = latest.get("status", "Unknown")
                                if status in ("Completed", "Succeeded"):
                                    print(f"  Refresh COMPLETED after {(poll_i+1)*10}s")
                                    success = True
                                    break
                                elif status == "Failed":
                                    err_msg = latest.get("serviceExceptionJson", "")
                                    print(f"  Refresh FAILED after {(poll_i+1)*10}s")
                                    if err_msg:
                                        try:
                                            err_obj = json.loads(err_msg)
                                            print(f"  Error: {err_obj.get('errorCode', '')}")
                                            print(f"  Detail: {err_obj.get('errorDescription', '')[:300]}")
                                        except Exception:
                                            print(f"  Error: {err_msg[:300]}")
                                    break
                            elif poll_i % 3 == 0:
                                print(f"  [{(poll_i+1)*10}s] Status: {status}...")
                    else:
                        print("  Refresh still running after 10 min. Check workspace.")
                        break

                    if success:
                        break
                    elif attempt < MAX_ATTEMPTS:
                        print(f"  Retrying in 20s...")
                        time.sleep(20)
                        token = notebookutils.credentials.getToken("pbi")
                        headers["Authorization"] = f"Bearer {token}"
                    else:
                        print("  All refresh attempts failed. Refresh manually from the workspace.")

print("=" * 60)


In [ ]:
# ============================================================================
# CELL 7 -- Deploy Ontology & Graph Model via Inline API Calls
# ============================================================================
# Deploys the ontology (10 entity types, 14 relationships) and a standalone
# graph model directly via the Fabric REST API -- no subprocess needed.
#
# Pattern from RTI-Hackathon-Demo Post_Deploy_Setup.ipynb:
#   - notebookutils token (no InteractiveBrowserCredential)
#   - Inline API calls (no subprocess)
#   - Delete-and-recreate for graph (updateDefinition unreliable in Preview)
#   - Token refresh before graph updateDefinition
#   - graph definition push (no updateMetadata -- just created fresh)
# ============================================================================

import json, requests, time, base64, os, re, math, uuid

print("=" * 60)
print("  ONTOLOGY & GRAPH MODEL -- API Deployment")
print("=" * 60)
print()

# -- Auth & Discovery -----------------------------------------
token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
API = "https://api.fabric.microsoft.com/v1"

print(f"  Workspace: {workspace_id}")

# Discover lh_gold_curated lakehouse ID
resp = requests.get(f"{API}/workspaces/{workspace_id}/lakehouses", headers=headers)
resp.raise_for_status()
lh_gold_id = None
for lh in resp.json().get("value", []):
    if lh["displayName"] == "lh_gold_curated":
        lh_gold_id = lh["id"]
        break
if not lh_gold_id:
    print("  [FAIL] lh_gold_curated not found -- run pipeline first")
    raise RuntimeError("lh_gold_curated not found")
print(f"  Lakehouse: lh_gold_curated ({lh_gold_id})")

# -- Paths -----------------------------------------------------
ONTOLOGY_NAME = "Healthcare_Demo_Ontology_HLS"
GRAPH_MODEL_NAME = "Healthcare_Demo_Graph"
ONT_API = f"{API}/workspaces/{workspace_id}/ontologies"
GM_API = f"{API}/workspaces/{workspace_id}/graphModels"

# Find the extracted repo -- launcher extracts ZIP to a subdirectory
# e.g. .lakehouse/default/Files/src/Fabric-Payer-Provider-HealthCare-Demo-main/
ont_dir = None
for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    if not os.path.isdir(base):
        continue
    # Check direct: base/ontology/ONTOLOGY_NAME
    direct = os.path.join(base, "ontology", ONTOLOGY_NAME)
    if os.path.isdir(direct):
        ont_dir = direct
        break
    # Check one level down (extracted repo subdirectory)
    for sub in os.listdir(base):
        nested = os.path.join(base, sub, "ontology", ONTOLOGY_NAME)
        if os.path.isdir(nested):
            ont_dir = nested
            break
    if ont_dir:
        break

if not ont_dir:
    print("  [FAIL] Ontology dir not found in extracted repo")
    print("  Searched: .lakehouse/default/Files/src[/<subdir>]/ontology/" + ONTOLOGY_NAME)
    # List what's actually at Files/src for debugging
    for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(base):
            print(f"  Contents of {base}: {os.listdir(base)[:10]}")
            for sub in os.listdir(base):
                subpath = os.path.join(base, sub)
                if os.path.isdir(subpath):
                    print(f"    {sub}/: {os.listdir(subpath)[:10]}")
    raise RuntimeError("Ontology dir not found")
print(f"  Ontology dir: {ont_dir}")

# -- Helper: LRO wait -----------------------------------------
def wait_lro(response, label, timeout=180):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            body = r.json()
            status = body.get("status", "")
            if status == "Succeeded":
                return True
            if status in ("Failed", "Cancelled"):
                err = body.get("error", {})
                err_code = err.get("code", "")
                err_msg = err.get("message", "")
                print(f"    [FAIL] {label}: {status}")
                if err_code or err_msg:
                    print(f"    Error: {err_code} - {err_msg[:500]}")
                else:
                    print(f"    Response: {json.dumps(body, indent=2)[:500]}")
                return False
    print(f"    [FAIL] {label}: timed out after {timeout}s")
    return False

# -- Helper: Load ontology parts from disk ---------------------
def load_ontology_parts(base_path):
    parts = []
    manifest_path = os.path.join(base_path, "manifest.json")
    if os.path.exists(manifest_path):
        with open(manifest_path, 'r', encoding='utf-8-sig') as f:
            manifest = json.load(f)
        for part_info in manifest.get("exportedParts", []):
            part_path = part_info["path"]
            file_path = os.path.join(base_path, part_path)
            if not os.path.exists(file_path):
                continue
            with open(file_path, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]
            parts.append({"path": part_path, "payload": base64.b64encode(raw).decode("utf-8"), "payloadType": "InlineBase64"})
    else:
        for root, _dirs, files in sorted(os.walk(base_path)):
            for fname in sorted(files):
                if fname in (".platform", "manifest.json"):
                    continue
                filepath = os.path.join(root, fname)
                rel_path = os.path.relpath(filepath, base_path).replace("\\", "/")
                with open(filepath, 'rb') as f:
                    raw = f.read()
                if raw.startswith(b'\xef\xbb\xbf'):
                    raw = raw[3:]
                parts.append({"path": rel_path, "payload": base64.b64encode(raw).decode("utf-8"), "payloadType": "InlineBase64"})
    return parts

# -- Helper: Patch data binding IDs ----------------------------
def patch_bindings(parts, ws_id, lh_id):
    patched = []
    for part in parts:
        path = part["path"]
        if "DataBindings/" in path or "Contextualizations/" in path:
            try:
                content = base64.b64decode(part["payload"]).decode("utf-8")
                obj = json.loads(content)
                _patch_ids(obj, ws_id, lh_id)
                content = json.dumps(obj, indent=2, ensure_ascii=False)
                part = {**part, "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8")}
            except Exception as e:
                print(f"    [WARN] Could not patch {path}: {e}")
        patched.append(part)
    return patched

def _patch_ids(obj, ws_id, lh_id):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            val = obj[key]
            lk = key.lower()
            if lk in ("workspaceid", "workspaceguid", "workspace_id"):
                obj[key] = ws_id
            elif lk in ("itemid", "lakehouseid", "artifactid", "item_id"):
                obj[key] = lh_id
            elif isinstance(val, str) and "onelake" in val.lower():
                m = re.match(r'(abfss://)([0-9a-f-]+)(@onelake[^/]*/)([0-9a-f-]+)(.*)', val, re.I)
                if m:
                    obj[key] = f"{m.group(1)}{ws_id}{m.group(3)}{lh_id}{m.group(5)}"
            elif isinstance(val, (dict, list)):
                _patch_ids(val, ws_id, lh_id)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                _patch_ids(item, ws_id, lh_id)

# ==============================================================
# STEP 1: DEPLOY ONTOLOGY
# ==============================================================
print()
print("Step 1: Deploy Ontology")
print("-" * 40)

# Load parts
parts = load_ontology_parts(ont_dir)
et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
rt_count = sum(1 for p in parts if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"))
print(f"  Loaded {len(parts)} parts: {et_count} entities, {rt_count} relationships")

# Patch data bindings
parts = patch_bindings(parts, workspace_id, lh_gold_id)
print(f"  Patched data bindings for target environment")

# Check if ontology already exists
ont_id = None
r = requests.get(ONT_API, headers=headers)
if r.status_code == 200:
    for o in r.json().get("value", []):
        if o["displayName"] == ONTOLOGY_NAME:
            ont_id = o["id"]
            print(f"  Already exists (ID: {ont_id}) -- will update definition")
            break

# Fallback: try items endpoint
if not ont_id and r.status_code != 200:
    r2 = requests.get(f"{API}/workspaces/{workspace_id}/items?type=Ontology", headers=headers)
    if r2.status_code == 200:
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break

if not ont_id:
    # Create new ontology
    create_body = {
        "displayName": ONTOLOGY_NAME,
        "description": f"Healthcare Ontology -- {et_count} entity types, {rt_count} relationships, bound to lh_gold_curated",
    }
    r = requests.post(ONT_API, headers=headers, json=create_body)
    if r.status_code in (200, 201):
        ont_id = r.json().get("id")
        print(f"  Created: {ont_id}")
    elif r.status_code == 202:
        wait_lro(r, "create ontology")
        time.sleep(3)
        r2 = requests.get(ONT_API, headers=headers)
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Created (async): {ont_id}")
    elif "AlreadyInUse" in (r.text or ""):
        r2 = requests.get(ONT_API, headers=headers)
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Found existing: {ont_id}")
    else:
        print(f"  [FAIL] Create ontology: HTTP {r.status_code} {r.text[:300]}")

ont_result = "[FAIL]"
if ont_id:
    # Push definition
    update_url = f"{ONT_API}/{ont_id}/updateDefinition"
    r = requests.post(update_url, headers=headers, json={"definition": {"parts": parts}})
    if r.status_code in (200, 201):
        print(f"  [OK] Definition pushed")
        ont_result = "[OK]"
    elif r.status_code == 202:
        ok = wait_lro(r, "updateDefinition")
        ont_result = "[OK]" if ok else "[FAIL]"
        print(f"  {'[OK]' if ok else '[FAIL]'} Definition push {'succeeded' if ok else 'failed'}")
    else:
        print(f"  [FAIL] updateDefinition: HTTP {r.status_code} {r.text[:300]}")

# ==============================================================
# STEP 2: DEPLOY GRAPH MODEL
# ==============================================================
print()
print("Step 2: Deploy Graph Model")
print("-" * 40)

# -- Load ontology metadata for graph definition --
entity_dir = os.path.join(ont_dir, "EntityTypes")
rel_dir = os.path.join(ont_dir, "RelationshipTypes")

TYPE_MAP = {"String": "STRING", "Int64": "INT64", "Double": "DOUBLE", "DateTime": "DATETIME",
            "Boolean": "BOOL", "Decimal": "DOUBLE", "Int32": "INT32", "Date": "DATETIME"}

entities = {}
for ent_name in sorted(os.listdir(entity_dir)):
    ent_path = os.path.join(entity_dir, ent_name)
    if not os.path.isdir(ent_path):
        continue
    defn_file = os.path.join(ent_path, "definition.json")
    if not os.path.exists(defn_file):
        continue
    with open(defn_file, 'r', encoding='utf-8-sig') as f:
        defn = json.load(f)
    eid = defn["id"]
    pk_prop_id = defn["entityIdParts"][0] if defn.get("entityIdParts") else None
    prop_map = {p["id"]: {"name": p["name"], "type": p["valueType"]} for p in defn.get("properties", [])}
    pk_name = prop_map[pk_prop_id]["name"] if pk_prop_id and pk_prop_id in prop_map else None

    bindings, source_table = {}, None
    db_dir = os.path.join(ent_path, "DataBindings")
    if os.path.isdir(db_dir):
        for fname in os.listdir(db_dir):
            if not fname.endswith(".json"):
                continue
            with open(os.path.join(db_dir, fname), 'r', encoding='utf-8-sig') as bf:
                db = json.load(bf)
            cfg = db.get("dataBindingConfiguration", {})
            bindings.update({pb["targetPropertyId"]: pb["sourceColumnName"] for pb in cfg.get("propertyBindings", [])})
            source_table = cfg.get("sourceTableProperties", {}).get("sourceTableName") or source_table

    entities[eid] = {"name": defn["name"], "pk": pk_name, "props": prop_map, "table": source_table, "bindings": bindings}

relationships = {}
if os.path.isdir(rel_dir):
    for rel_name in sorted(os.listdir(rel_dir)):
        rel_path = os.path.join(rel_dir, rel_name)
        if not os.path.isdir(rel_path):
            continue
        defn_file = os.path.join(rel_path, "definition.json")
        if not os.path.exists(defn_file):
            continue
        with open(defn_file, 'r', encoding='utf-8-sig') as f:
            defn = json.load(f)
        rid = defn["id"]
        src_id, tgt_id = defn["source"]["entityTypeId"], defn["target"]["entityTypeId"]

        ctx_table, src_cols, tgt_cols = None, [], []
        ctx_dir = os.path.join(rel_path, "Contextualizations")
        if os.path.isdir(ctx_dir):
            for fname in os.listdir(ctx_dir):
                if not fname.endswith(".json"):
                    continue
                with open(os.path.join(ctx_dir, fname), 'r', encoding='utf-8-sig') as cf:
                    ctx = json.load(cf)
                ctx_table = ctx.get("dataBindingTable", {}).get("sourceTableName")
                src_cols = [sk["sourceColumnName"] for sk in ctx.get("sourceKeyRefBindings", [])]
                tgt_cols = [tk["sourceColumnName"] for tk in ctx.get("targetKeyRefBindings", [])]

        relationships[rid] = {"name": defn["name"], "src": src_id, "tgt": tgt_id,
                              "table": ctx_table, "src_cols": src_cols, "tgt_cols": tgt_cols}

valid_ids = {eid for eid, e in entities.items() if e["table"]}
print(f"  Loaded: {len(entities)} entities ({len(valid_ids)} valid), {len(relationships)} relationships")

# -- Build 5-part graph definition -----------------------------
# 1. graphType.json
node_types = []
for eid, e in entities.items():
    if eid not in valid_ids:
        continue
    props = [{"name": p["name"], "type": TYPE_MAP.get(p["type"], "STRING")}
             for pid, p in e["props"].items() if pid in e["bindings"]]
    node_types.append({"alias": f"{e['name']}_nodeType", "labels": [e["name"]],
                       "primaryKeyProperties": [e["pk"]] if e["pk"] else [], "properties": props})

edge_types = []
for rid, r in relationships.items():
    if not r["table"] or not r["src_cols"] or not r["tgt_cols"]:
        continue
    if r["src"] not in valid_ids or r["tgt"] not in valid_ids:
        continue
    edge_types.append({
        "alias": f"{r['name']}_edgeType", "labels": [r["name"]],
        "sourceNodeType": {"alias": f"{entities[r['src']]['name']}_nodeType"},
        "destinationNodeType": {"alias": f"{entities[r['tgt']]['name']}_nodeType"},
        "properties": []
    })

graph_type = {"schemaVersion": "1.0.0", "nodeTypes": node_types, "edgeTypes": edge_types}
print(f"    graphType: {len(node_types)} nodes, {len(edge_types)} edges")

# 2. graphDefinition.json
node_tables = [{"id": f"{e['name']}_{uuid.uuid4().hex[:8]}", "nodeTypeAlias": f"{e['name']}_nodeType",
                "dataSourceName": f"{e['table']}_Source",
                "propertyMappings": [{"propertyName": p["name"], "sourceColumn": e["bindings"][pid]}
                                     for pid, p in e["props"].items() if pid in e["bindings"]]}
               for eid, e in entities.items() if eid in valid_ids]

edge_tables = [{"id": f"{r['name']}_{uuid.uuid4().hex[:8]}", "edgeTypeAlias": f"{r['name']}_edgeType",
                "dataSourceName": f"{r['table']}_Source",
                "sourceNodeKeyColumns": r["src_cols"], "destinationNodeKeyColumns": r["tgt_cols"],
                "propertyMappings": []}
               for rid, r in relationships.items()
               if r["table"] and r["src_cols"] and r["tgt_cols"] and r["src"] in valid_ids and r["tgt"] in valid_ids]

graph_def = {"schemaVersion": "1.0.0", "nodeTables": node_tables, "edgeTables": edge_tables}
print(f"    graphDefinition: {len(node_tables)} nodeTables, {len(edge_tables)} edgeTables")

# 3. dataSources.json -- discover actual table paths via Spark
tables = {e["table"] for eid, e in entities.items() if eid in valid_ids and e["table"]}
tables |= {r["table"] for r in relationships.values()
           if r["table"] and r["src_cols"] and r["tgt_cols"] and r["src"] in valid_ids and r["tgt"] in valid_ids}

table_paths = {}
for t in sorted(tables):
    try:
        loc_df = spark.sql(f"DESCRIBE EXTENDED lh_gold_curated.{t}")
        for row in loc_df.collect():
            if row['col_name'] == 'Location':
                table_paths[t] = row['data_type']
                break
        if t not in table_paths:
            raise Exception("Location not found")
    except Exception:
        table_paths[t] = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lh_gold_id}/Tables/{t}"

data_sources = {"dataSources": [{"name": f"{t}_Source", "type": "DeltaTable",
                "properties": {"path": table_paths[t]}} for t in sorted(tables)]}
print(f"    dataSources: {len(tables)} tables")

# 4. stylingConfiguration.json
n = len(node_types)
radius = 300
positions = {}
for i, nt in enumerate(node_types):
    alias = nt["alias"]
    positions[alias] = {"x": int(radius + radius * math.cos(2 * math.pi * i / max(n, 1))),
                        "y": int(radius + radius * math.sin(2 * math.pi * i / max(n, 1)))}
styling = {"schemaVersion": "1.0.0", "modelLayout": {"positions": positions,
           "styles": {k: {"size": 30} for k in positions}, "pan": {"x": 0, "y": 0}, "zoomLevel": 1}}


def encode_part(path, content):
    return {"path": path, "payload": base64.b64encode(json.dumps(content, indent=2).encode()).decode(), "payloadType": "InlineBase64"}

gm_parts = [encode_part("graphType.json", graph_type), encode_part("graphDefinition.json", graph_def),
            encode_part("dataSources.json", data_sources), encode_part("stylingConfiguration.json", styling)]
print(f"  Total: {len(gm_parts)} definition parts (parts)")

# -- Delete existing graph (delete-and-recreate is more reliable) --
print()
deleted_any = False
r = requests.get(GM_API, headers=headers)
if r.status_code == 200:
    for g in r.json().get("value", []):
        gn = g.get("displayName", "")
        if gn == GRAPH_MODEL_NAME or ONTOLOGY_NAME in gn:
            gid = g["id"]
            print(f"  Deleting existing graph: {gn}...")
            dr = requests.delete(f"{GM_API}/{gid}", headers=headers)
            if dr.status_code in (200, 202, 204):
                if dr.status_code == 202:
                    wait_lro(dr, "delete graph")
                print(f"    Deleted")
                deleted_any = True

if deleted_any:
    print("  Waiting 30s for name availability...")
    time.sleep(30)

# -- Create fresh graph (with retry) --
print(f"  Creating graph: {GRAPH_MODEL_NAME}...")
gm_id = None
gm_success = False

for attempt in range(3):
    # Refresh token before each attempt
    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    r = requests.post(GM_API, headers=headers, json={
        "displayName": GRAPH_MODEL_NAME,
        "description": f"Graph for {ONTOLOGY_NAME}. {len(node_types)} nodes, {len(edge_types)} edges."
    })
    if r.status_code in (200, 201):
        gm_id = r.json().get("id")
        print(f"  Created: {gm_id}")
        break
    elif r.status_code == 202:
        ok = wait_lro(r, "create graph")
        r2 = requests.get(GM_API, headers=headers)
        for g in r2.json().get("value", []):
            if g["displayName"] == GRAPH_MODEL_NAME:
                gm_id = g["id"]
                break
        if gm_id:
            print(f"  Created (async): {gm_id}")
            break
    elif r.status_code == 409 or "NotAvailableYet" in (r.text or ""):
        print(f"  Name not available yet, waiting 15s... (attempt {attempt + 1}/3)")
        time.sleep(15)
    else:
        print(f"  [FAIL] Create graph: HTTP {r.status_code} {r.text[:200]}")
        break

# -- Push definition (with retry) --
if gm_id:
    print(f"  Waiting 180s for graph backend hydration (eventual consistency)...")
    time.sleep(180)

    url = f"{GM_API}/{gm_id}/updateDefinition"

    for defn_attempt in range(3):
        # Refresh token before each attempt
        token = notebookutils.credentials.getToken("pbi")
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

        # Debug: show what we are sending
        if defn_attempt == 0:
            print(f"    Parts: {[p['path'] for p in gm_parts]}")
            for p in gm_parts:
                import base64 as _b64
                _content = _b64.b64decode(p["payload"]).decode("utf-8")
                _preview = _content[:200].replace(chr(10), " ")
                print(f"    {p['path']}: {len(_content)} bytes -- {_preview}...")
        print(f"  Pushing definition (attempt {defn_attempt + 1}/3)...")
        r = requests.post(url, headers=headers, json={"definition": {"format": "json", "parts": gm_parts}})

        if r.status_code in (200, 201):
            gm_success = True
            print(f"  [OK] Definition pushed")
            break
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition", timeout=300)
            if ok:
                gm_success = True
                print(f"  [OK] Definition push")
                break
            else:
                print(f"  Definition push failed on attempt {defn_attempt + 1}")
                if defn_attempt < 2:
                    wait_time = 60 * (defn_attempt + 1)
                    print(f"  Retrying in {wait_time}s...")
                    time.sleep(wait_time)
        else:
            print(f"  [FAIL] updateDefinition: HTTP {r.status_code} {r.text[:500]}")
            if defn_attempt < 2:
                wait_time = 60 * (defn_attempt + 1)
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)

    if not gm_success:
        print(f"  [FAIL] All updateDefinition attempts failed")
        print(f"  You can manually load the graph from the Fabric UI.")

# -- Wait for data load --
if gm_id and gm_success:
    print(f"  Waiting for data load...")
    for poll in range(60):
        r = requests.get(f"{GM_API}/{gm_id}", headers=headers)
        if r.status_code == 200:
            props = r.json().get("properties", {})
            readiness = props.get("queryReadiness", "")
            status = (props.get("lastDataLoadingStatus") or {}).get("status", "")
            if readiness == "Full" or status == "Completed":
                print(f"  [OK] Data loaded -- queryReadiness={readiness}")
                break
            if status == "Failed":
                print(f"  [WARN] Data load failed -- check Fabric UI")
                break
            if poll % 6 == 0:
                print(f"    readiness={readiness}, status={status}...")
        time.sleep(5)

# -- Summary --
graph_result = "[OK]" if gm_success else "[FAIL]"
print()
print("=" * 60)
print(f"  ONTOLOGY:    {ONTOLOGY_NAME:<38} {ont_result}")
print(f"  GRAPH MODEL: {GRAPH_MODEL_NAME:<38} {graph_result}")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 7b -- Deploy Real-Time Intelligence (RTI)
# ============================================================================
# Eventhouse + KQL Database are already deployed as Git artifacts (Stage 2).
# This cell runs post-deploy wiring and scoring notebooks:
#   1. NB_RTI_Setup_Eventhouse  -- Discovers artifacts, creates KQL tables, discovers Kusto URI
#   2. NB_RTI_Event_Simulator   -- Generates initial events (batch mode) + pushes to KQL
#   3. NB_RTI_Fraud_Detection   -- Scores claims for fraud -> Delta + KQL
#   4. NB_RTI_Care_Gap_Alerts   -- Point-of-care gap closure alerts -> Delta + KQL
#   5. NB_RTI_HighCost_Trajectory -- High-cost trajectory analysis -> Delta + KQL
# ============================================================================
# Ingestion: Direct Kusto (azure-kusto-ingest, pre-installed in Fabric)
# No Eventstream or connection strings needed -- zero config for users.
# ============================================================================

if DEPLOY_RTI:
    print("=" * 60)
    print("  REAL-TIME INTELLIGENCE DEPLOYMENT")
    print("=" * 60)

    # -- Attach lh_gold_curated to RTI notebooks -------------------------
    # Notebooks run via notebookutils.notebook.run() do not inherit the
    # caller's lakehouse context.  The child notebook MUST have a default
    # lakehouse in its metadata, otherwise Fabric blocks ALL Spark SQL
    # with "No default context found".
    # We patch each notebook's ipynb definition here before running them.
    # ---------------------------------------------------------------------
    import requests, base64, time as _time

    _token = notebookutils.credentials.getToken("pbi")
    _hdrs = {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}
    _api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Get all workspace items
    _items_resp = requests.get(f"{_api}/items", headers=_hdrs)
    _name_to_id = {(it["type"], it["displayName"]): it["id"] for it in _items_resp.json().get("value", [])}
    _lh_id = _name_to_id.get(("Lakehouse", "lh_gold_curated"), "")

    _lh_deps = {
        "lakehouse": {
            "default_lakehouse": _lh_id,
            "default_lakehouse_name": "lh_gold_curated",
            "default_lakehouse_workspace_id": workspace_id,
            "known_lakehouses": [
                {"id": _lh_id, "displayName": "lh_gold_curated", "isDefault": True}
            ],
        }
    }

    _needs_lh = ["NB_RTI_Setup_Eventhouse", "NB_RTI_Event_Simulator", "NB_RTI_Fraud_Detection",
                 "NB_RTI_Care_Gap_Alerts", "NB_RTI_HighCost_Trajectory"]

    def _get_nb_definition(nb_id, hdrs, api_base):
        """Get notebook definition in ipynb format, handling LRO."""
        # Use notebook-specific endpoint with explicit ipynb format
        url = f"{api_base}/notebooks/{nb_id}/getDefinition?format=ipynb"
        r = requests.post(url, headers=hdrs)
        print(f"      getDefinition: HTTP {r.status_code}")

        if r.status_code == 200:
            return r.json()

        if r.status_code != 202:
            print(f"      Unexpected status: {r.text[:200]}")
            return None

        # LRO - poll operation status
        loc = r.headers.get("Location", "")
        retry_secs = int(r.headers.get("Retry-After", "2"))
        if not loc:
            print(f"      No Location header in 202 response")
            return None

        print(f"      LRO polling ({retry_secs}s interval)...")
        for attempt in range(60):
            _time.sleep(retry_secs)
            poll_r = requests.get(loc, headers=hdrs)
            if poll_r.status_code == 202:
                continue
            if poll_r.status_code != 200:
                print(f"      LRO poll: HTTP {poll_r.status_code}")
                return None

            body = poll_r.json()
            status = body.get("status", "")
            if status == "Running" or status == "NotStarted":
                continue
            if status != "Succeeded":
                print(f"      LRO status: {status}")
                return None

            # LRO succeeded - fetch the actual result
            # Try {loc}/result FIRST (most reliable)
            result_url = f"{loc}/result"
            print(f"      LRO succeeded, fetching /result...")
            result_r = requests.get(result_url, headers=hdrs)
            if result_r.status_code == 200:
                result_body = result_r.json()
                parts = result_body.get("definition", {}).get("parts", [])
                if not parts:
                    parts = result_body.get("parts", [])
                if parts:
                    print(f"      Got {len(parts)} parts from /result")
                    return result_body

            # Try resourceLocation (sometimes points to item URL, not definition)
            res_loc = body.get("resourceLocation", "")
            if res_loc and res_loc != result_url:
                print(f"      Trying resourceLocation...")
                res_r = requests.get(res_loc, headers=hdrs)
                if res_r.status_code == 200:
                    res_body = res_r.json()
                    parts = res_body.get("definition", {}).get("parts", [])
                    if not parts:
                        parts = res_body.get("parts", [])
                    if parts:
                        print(f"      Got {len(parts)} parts from resourceLocation")
                        return res_body

            # Try the poll body itself (some APIs embed result in final status)
            parts = body.get("definition", {}).get("parts", [])
            if not parts:
                parts = body.get("parts", [])
            if parts:
                print(f"      Got {len(parts)} parts from poll body")
                return body

            print(f"      WARNING: LRO succeeded but no parts found in any endpoint")
            return None

        print(f"      LRO timed out after 60 polls")
        return None

    if _lh_id:
        print(f"\n  Attaching lh_gold_curated ({_lh_id[:8]}...) to RTI notebooks...")
        for _nb_name in _needs_lh:
            _nb_id = _name_to_id.get(("Notebook", _nb_name))
            if not _nb_id:
                print(f"    {_nb_name}: not found in workspace, skipping")
                continue

            print(f"    {_nb_name}:")
            result = _get_nb_definition(_nb_id, _hdrs, _api)

            if result is None:
                print(f"      SKIP - could not get definition")
                continue

            # Extract parts from response (handle both nesting patterns)
            _defn = result.get("definition", result)
            _parts = _defn.get("parts", [])

            if not _parts:
                print(f"      SKIP - 0 parts in response")
                print(f"      Response keys: {list(result.keys())}")
                if "definition" in result:
                    print(f"      definition keys: {list(result['definition'].keys())}")
                continue

            print(f"      Parts: {[p.get('path','?') for p in _parts]}")

            # Find and patch the notebook content part
            _patched = False
            for _part in _parts:
                _path = _part.get("path", "")
                # Match ipynb content parts (various naming patterns)
                if _path.endswith(".ipynb") or "content" in _path.lower() or _path == "artifact.content.ipynb":
                    try:
                        _raw = base64.b64decode(_part["payload"]).decode("utf-8")
                        _nb_json = json.loads(_raw)
                        # Patch metadata with lakehouse dependency
                        _meta = _nb_json.setdefault("metadata", {})
                        _meta["dependencies"] = _lh_deps
                        _part["payload"] = base64.b64encode(
                            json.dumps(_nb_json).encode("utf-8")
                        ).decode("utf-8")
                        _patched = True
                        print(f"      Patched lakehouse into {_path}")
                    except json.JSONDecodeError:
                        # Not JSON (probably .py format) -- skip this part
                        print(f"      {_path} is not JSON, trying next part...")
                        continue
                    except Exception as _ex:
                        print(f"      Failed to patch {_path}: {_ex}")
                        continue
                    break

            if not _patched:
                print(f"      Could not patch any part (no ipynb content found)")
                # As a last resort, try to find ANY JSON part we can decode
                for _part in _parts:
                    _path = _part.get("path", "")
                    if _path == ".platform":
                        continue
                    try:
                        _raw = base64.b64decode(_part["payload"]).decode("utf-8")
                        _nb_json = json.loads(_raw)
                        if "cells" in _nb_json or "nbformat" in _nb_json:
                            _meta = _nb_json.setdefault("metadata", {})
                            _meta["dependencies"] = _lh_deps
                            _part["payload"] = base64.b64encode(
                                json.dumps(_nb_json).encode("utf-8")
                            ).decode("utf-8")
                            _patched = True
                            print(f"      Patched lakehouse into {_path} (by content detection)")
                            break
                    except Exception:
                        continue

            if not _patched:
                print(f"      SKIP - no patchable content part found")
                continue

            # Push updated definition back
            _update_body = {"definition": {"format": "ipynb", "parts": _parts}}
            _ur = requests.post(
                f"{_api}/notebooks/{_nb_id}/updateDefinition",
                headers=_hdrs,
                json=_update_body,
            )
            if _ur.status_code == 200:
                print(f"      Lakehouse attached (200)")
            elif _ur.status_code == 202:
                _uloc = _ur.headers.get("Location", "")
                if _uloc:
                    for _ in range(30):
                        _time.sleep(2)
                        _pr = requests.get(_uloc, headers=_hdrs)
                        if _pr.status_code != 202:
                            break
                print(f"      Lakehouse attached (202->done)")
            else:
                print(f"      updateDefinition failed: HTTP {_ur.status_code} {_ur.text[:300]}")
    else:
        print("  WARNING: lh_gold_curated not found in workspace -- cannot attach lakehouse")

    # -- Run RTI notebooks ------------------------------------------------
    rti_notebooks = [
        "NB_RTI_Setup_Eventhouse",
        "NB_RTI_Event_Simulator",
        "NB_RTI_Fraud_Detection",
        "NB_RTI_Care_Gap_Alerts",
        "NB_RTI_HighCost_Trajectory",
    ]

    for nb_name in rti_notebooks:
        print(f"\n  Running {nb_name}...")
        try:
            notebookutils.notebook.run(nb_name, 1200)
            print(f"  -> {nb_name}: OK")
        except Exception as e:
            print(f"  -> {nb_name}: FAILED -- {e}")
            print(f"    You can run this notebook manually from the workspace.")

    print("\n" + "=" * 60)
    print("  RTI DEPLOYMENT COMPLETE")
    print("=" * 60)
    print("  Delta tables in lh_gold_curated:")
    print("    - rti_claims_events, rti_adt_events, rti_rx_events")
    print("    - rti_fraud_scores, rti_care_gap_alerts, rti_highcost_alerts")
    print()
    print("  KQL tables in Healthcare_RTI_Eventhouse:")
    print("    - claims_events, adt_events, rx_events")
    print("    - fraud_scores, care_gap_alerts, highcost_alerts")
    print()
    print("  Ingestion: Direct Kusto (zero config)")
    print("  For streaming: set MODE='stream' in NB_RTI_Event_Simulator")
    print("=" * 60)
else:
    print("Skipping RTI deployment (DEPLOY_RTI=False)")


In [ ]:
# ============================================================================
# CELL 8 — Deployment Summary
# ============================================================================

import requests

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# Group by type
by_type = {}
for item in items:
    t = item.get("type", "Unknown")
    by_type.setdefault(t, []).append(item["displayName"])

print("=" * 60)
print("  DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Workspace: {workspace_id}")
print(f"  Total items: {len(items)}")
print()
for item_type in ["Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"]:
    names = by_type.get(item_type, [])
    if names:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# Show any other types
shown = {"Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"}
for item_type, names in by_type.items():
    if item_type not in shown:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# RTI table counts
print()
print("  RTI Delta Tables (lh_gold_curated):")
try:
    for tbl in ["rti_claims_events", "rti_adt_events", "rti_rx_events", "rti_fraud_scores", "rti_care_gap_alerts", "rti_highcost_alerts"]:
        try:
            cnt = spark.table(tbl).count()
            print(f"    - {tbl}: {cnt:,} rows")
        except Exception:
            print(f"    - {tbl}: (not created yet)")
except Exception:
    print("    (could not query RTI tables)")

print()
print("=" * 60)
print("  NEXT STEPS")
print("=" * 60)
print("  1. Open the HealthcareDemoHLS semantic model -> verify tables loaded")
print("  2. Open the HealthcareHLSAgent -> ask: 'What are the top denial reasons?'")
print("  3. Verify the Ontology + Graph Model (deployed after pipeline)")
print("     -> Open Healthcare_Demo_Ontology_HLS in workspace to explore")
print("  4. For incremental loads: run PL_Healthcare_Master with load_mode=incremental")
print("     (auto-generates incremental data + runs full ETL in one shot)")
print("  5. RTI: Open Healthcare_RTI_Eventhouse to explore real-time data")
print("     - NB_RTI_Fraud_Detection: View rti_fraud_scores for flagged claims")
print("     - NB_RTI_Care_Gap_Alerts: View rti_care_gap_alerts for gap closures")
print("     - NB_RTI_HighCost_Trajectory: View rti_highcost_alerts for cost trends")
print("  6. For live streaming: set MODE='stream' in NB_RTI_Event_Simulator")
print("     and configure the Eventstream Custom App connection string")
print("=" * 60)